# SQL database & analytical queries

I load the cleaned dataset into a SQLite database and run five analytical 
queries.

**Database:** SQLite (`supply_chain.db`)  
**Table:** `orders` (180,519 rows · 55 columns)  
**Tool:** Python `sqlite3` + `pandas.read_sql`

In [1]:
import sqlite3
import pandas as pd

df = pd.read_csv('../data/supply_chain_clean.csv',parse_dates=['order_date_dateorders', 'shipping_date_dateorders'], low_memory=False)

conn = sqlite3.connect('../data/supply_chain.db')

df.to_sql('orders', conn, if_exists='replace', index=False)

verify = pd.read_sql('SELECT COUNT(*) as total_rows FROM orders', conn)
print(f"Database ready. Rows loaded: {verify.iloc[0,0]:,}")

Database ready. Rows loaded: 180,519


In [2]:
q1 = """
SELECT 
    shipping_mode,
    COUNT(*)                                    AS total_orders,
    ROUND(AVG(late_delivery_risk) * 100, 1)    AS late_rate_pct,
    ROUND(AVG(shipping_delay), 2)              AS avg_delay_days,
    ROUND(SUM(sales), 2)                       AS total_revenue
FROM orders
GROUP BY shipping_mode
ORDER BY late_rate_pct DESC
"""
pd.read_sql(q1, conn)

,shipping_mode,total_orders,late_rate_pct,avg_delay_days,total_revenue
0,First Class,27814,95.3,1.00,5674369.76
1,Second Class,35216,76.6,1.99,7145444.82
2,Same Day,9737,45.7,0.48,1942528.56
3,Standard Class,107752,38.1,-0.00,22022391.88


## ABC product classification

Using a CTE and window function to rank products by cumulative revenue share.
Products are classified as:
- **A** — top 80% of revenue (focus inventory here)
- **B** — next 15% (monitor)
- **C** — bottom 5% (candidates for reduction)

In [3]:
q2 = """
WITH revenue_by_product AS (
    SELECT 
        product_name,
        category_name,
        SUM(sales) AS total_revenue,
        SUM(benefit_per_order)  AS total_profit
    FROM orders
    GROUP BY product_name, category_name
),
ranked AS (
    SELECT *,
        SUM(total_revenue) OVER (ORDER BY total_revenue DESC) AS cumulative_revenue,
        SUM(total_revenue) OVER () AS grand_total
    FROM revenue_by_product
)
SELECT 
    product_name,
    category_name,
    ROUND(total_revenue, 2) AS total_revenue,
    ROUND(cumulative_revenue / grand_total * 100, 1) AS cumulative_pct,
    CASE 
        WHEN cumulative_revenue / grand_total <= 0.80 THEN 'A'
        WHEN cumulative_revenue / grand_total <= 0.95 THEN 'B'
        ELSE 'C'
    END AS abc_class
FROM ranked
ORDER BY total_revenue DESC
LIMIT 25
"""
abc = pd.read_sql(q2, conn)
abc

,product_name,category_name,total_revenue,cumulative_pct,abc_class
0,Field & Stream Sportsman 16 Gun Fire Safe,Fishing,6929653.69,18.8,A
1,Perfect Fitness Perfect Rip Deck,Cleats,4421143.14,30.9,A
2,Diamondback Women's Serene Classic Comfort Bi,Camping & Hiking,4118425.57,42.1,A
3,Nike Men's Free 5.0+ Running Shoe,Cardio Equipment,3667633.20,52.0,A
4,Nike Men's Dri-FIT Victory Golf Polo,Women's Apparel,3147800.00,60.6,A
5,Pelican Sunstream 100 Kayak,Water Sports,3099845.09,69.0,A
6,Nike Men's CJ Elite 2 TD Football Cleat,Men's Footwear,2891757.66,76.9,A
7,O'Brien Men's Neoprene Life Vest,Indoor/Outdoor Games,2888993.91,84.7,B
8,Under Armour Girls' Toddler Spine Surge Runni,Shop By Sport,1269082.67,88.2,B
9,Dell Laptop,Computers,663000.00,90.0,B


### Findings

The ABC analysis confirms the Pareto principle holds in this dataset.
Class A products are the inventory priority for stockout prevention.
Class C products are candidates for safety stock reduction.

## Revenue at risk from late deliveries by category

We quantify the financial exposure of late deliveries per category, not just the rate, but the actual dollar value affected.

In [4]:
q3 = """
SELECT 
    category_name,
    COUNT(*)                                                            AS total_orders,
    SUM(late_delivery_risk)                                            AS late_orders,
    ROUND(AVG(late_delivery_risk) * 100, 1)                           AS late_rate_pct,
    ROUND(SUM(CASE WHEN late_delivery_risk = 1 THEN sales ELSE 0 END), 2) AS revenue_at_risk
FROM orders
GROUP BY category_name
ORDER BY revenue_at_risk DESC
LIMIT 10
"""
pd.read_sql(q3, conn)

,category_name,total_orders,late_orders,late_rate_pct,revenue_at_risk
0,Fishing,17325,9516,54.9,3806209.78
1,Cleats,24551,13496,55.0,2428216.22
2,Camping & Hiking,13729,7487,54.5,2245950.34
3,Cardio Equipment,12487,6805,54.5,2007340.74
4,Women's Apparel,21035,11476,54.6,1719450.00
5,Water Sports,15540,8517,54.8,1707064.88
6,Indoor/Outdoor Games,19298,10565,54.7,1583466.35
7,Men's Footwear,22246,12121,54.5,1575608.86
8,Shop By Sport,10984,6058,55.2,721384.43
9,Computers,442,224,50.7,336000.00


In [5]:
q4 = """
WITH segment_stats AS (
    SELECT 
        customer_segment,
        COUNT(*)                        AS order_count,
        ROUND(SUM(sales), 2)           AS total_revenue,
        ROUND(SUM(benefit_per_order), 2) AS total_profit,
        ROUND(AVG(profit_margin) * 100, 2) AS avg_margin_pct
    FROM orders
    GROUP BY customer_segment
)
SELECT *,
    RANK() OVER (ORDER BY total_profit DESC) AS profit_rank
FROM segment_stats
ORDER BY profit_rank
"""
pd.read_sql(q4, conn)

,customer_segment,order_count,total_revenue,total_profit,avg_margin_pct,profit_rank
0,Consumer,93504,19095790.16,2073487.67,12.10,1
1,Corporate,54789,11168406.84,1202574.96,12.07,2
2,Home Office,32226,6520538.02,690840.34,11.84,3


In [6]:
q5 = """
SELECT 
    order_year,
    order_month,
    COUNT(*)                         AS order_count,
    ROUND(SUM(sales), 2)            AS total_revenue,
    ROUND(SUM(benefit_per_order), 2) AS total_profit,
    ROUND(AVG(profit_margin) * 100, 2) AS avg_margin_pct
FROM orders
WHERE order_year < 2018
  AND NOT (order_year = 2017 AND order_month >= 9)
GROUP BY order_year, order_month
ORDER BY order_year, order_month
"""
monthly = pd.read_sql(q5, conn)
monthly

,order_year,order_month,order_count,total_revenue,total_profit,avg_margin_pct
0,2015,1,5322,1051590.08,111660.74,11.58
1,2015,2,4729,927009.90,99140.66,11.92
2,2015,3,5362,1051253.69,113778.21,12.67
3,2015,4,5126,1014463.28,108083.68,11.05
4,2015,5,5357,1050478.44,112147.90,12.12
5,2015,6,5134,1024006.17,110147.16,12.18
6,2015,7,5299,1038081.19,115624.06,12.22
7,2015,8,5273,1029494.69,117979.77,13.11
8,2015,9,5140,1018338.60,113467.94,12.44
9,2015,10,5302,1049154.27,101757.87,11.60


In [8]:
queries = {
    "shipping_performance": q1,
    "abc_classification": q2,
    "revenue_at_risk": q3,
    "segment_profitability": q4,
    "monthly_revenue_trend": q5,
}

with open('../sql/supply_chain_queries.sql', 'w') as f:
    for name, query in queries.items():
        f.write(f"-- {name.replace('_', ' ').upper()}\n")
        f.write(query.strip())
        f.write("\n\n")

print("Saved to sql/supply_chain_queries.sql")

Saved to sql/supply_chain_queries.sql


---

## Summary

| Query | Key output |
|-------|-----------|
| Shipping performance | First Class: 95.3% late rate, highest revenue exposure |
| ABC classification | Class A = ~20% of products driving 80% of revenue |
| Revenue at risk | [fill in your top category and dollar amount] |
| Customer segments | [fill in your #1 segment by profit] |
| Monthly trend | Stable ~5,200 orders/month, truncated Sep 2017 onward |
